In [1]:
from pymongo import MongoClient

MONGO_URI = "mongodb://10.255.68.40:27017"  # 인증 있으면 여기에 반영
DB_NAME = "ejoow"
SRC_COL = "2. US_sequence"

client = MongoClient(MONGO_URI, serverSelectionTimeoutMS=5000)
db = client[DB_NAME]
src = db[SRC_COL]

print("Connected OK:", client.admin.command("ping"))
print("Count (estimated):", src.estimated_document_count())


Connected OK: {'ok': 1.0}
Count (estimated): 713478


In [3]:
from pymongo import MongoClient

MONGO_URI = "mongodb://10.255.68.40:27017"
DB_NAME = "ejoow"
SRC_COL = "2. US_sequence"
DST_COL = "2. US_sequence_catlen_gt1"

client = MongoClient(MONGO_URI)
db = client[DB_NAME]
src = db[SRC_COL]

print("ping:", client.admin.command("ping"))
print("src estimated:", src.estimated_document_count())


ping: {'ok': 1.0}
src estimated: 713478


In [4]:
pipeline = [
    {
        "$match": {
            "$expr": {
                "$gt": [
                    {"$size": "$category_sequence"},
                    1
                ]
            }
        }
    },
    {
        "$merge": {
            "into": DST_COL,
            "whenMatched": "replace",
            "whenNotMatched": "insert"
        }
    }
]

src.aggregate(pipeline, allowDiskUse=True)
print("done. dst estimated:", db[DST_COL].estimated_document_count())


done. dst estimated: 113759


In [5]:
# 길이 1 문서가 없는지 확인
db[DST_COL].find_one({
    "$expr": {"$lte": [{"$size": "$category_sequence"}, 1]}
})
